# HypeTune_Py — Reusable Template

Drop in any binary-classification table and run the same inspect → importance → GridSearchCV → RandomizedSearchCV → simulation loop.

Edit the **CONFIG** cell, then run all.


## Inline cheat-sheet (keep this cell visible)

See also **`HypeTune_Py_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Parameter vs hyperparameter | Coefficients / split thresholds are **learned**. `max_depth`, `C`, `penalty`, `k` are **chosen**. |
| Gini importance | `clf.feature_importances_` after a tree/forest fit with `criterion='gini'`. Biased toward high-cardinality numeric features; ignores correlation. |
| Permutation importance | Shuffle one column, measure drop in score. Model-agnostic, needs a held-out set, slower. `sklearn.inspection.permutation_importance`. |
| Scaled \|coef\| | Only comparable after `StandardScaler`. L1 can zero features. |
| Grid search | Exhaustive Cartesian product of *lists*. `GridSearchCV(est, param_grid, cv=5)`. |
| Random search | Sample `n_iter` draws from *distributions*. `RandomizedSearchCV(est, param_distributions, n_iter=8)`. |
| `uniform(loc, scale)` | Draws on `[loc, loc+scale]`. `uniform(0, 100)` → C ∈ [0, 100]. |
| Attributes after `.fit` | `.best_estimator_`, `.best_params_`, `.best_score_` (mean CV), `.cv_results_`, `.score(X_test, y_test)`. |
| Never | Report `.best_score_` as the final generalisation number. That fold was used to *pick* the hyperparams. |
| `liblinear` | Needed if you still pass `penalty='l1'` on `LogisticRegression`. sklearn ≥ 1.8 prefers `l1_ratio` (0 = L2, 1 = L1). |
| Split once | Freeze `random_state`. Do not retune on the test fold. |


## Flowchart of the desired outcome

![HypeTune flow](hypetune_flowchart.png)

Inspect the balanced 900-row card → estimate which morphometrics actually move the class → exhaust a small tree grid → sample a continuous `C` for logistic regression → confirm on the hold-out fold → poke the knobs in the simulation cell.


## CONFIG — edit this cell for a new table

In [ ]:
TARGET = "Class"
DATA_PATH = "data/Raisin_Dataset.csv"   # or any CSV with TARGET as a column
TEST_SIZE = 0.25
RANDOM_STATE = 19

TREE_GRID = {"max_depth": [3, 5, 7], "min_samples_split": [2, 3, 4]}
LR_DIST = None  # filled after scipy import
N_ITER = 8
CV = 5


## Packages + load + split

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from scipy.stats import uniform, loguniform

np.random.seed(19)
plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

from scipy.stats import uniform
LR_DIST = {"penalty": ["l1", "l2"], "C": uniform(0, 100)}

df = pd.read_csv(DATA_PATH)
assert TARGET in df.columns, TARGET
X = df.drop(columns=TARGET)
y = df[TARGET]
print(df.shape, y.value_counts().to_dict())
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y if y.nunique()==2 else None
)


## Importance snapshot

In [ ]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE).fit(X_train, y_train)
print("DT Gini\n", pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False).round(3))
print("RF Gini\n", pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).round(3))


## Grid search (tree) + random search (logistic)

In [ ]:
grid = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), TREE_GRID, cv=CV)
grid.fit(X_train, y_train)
print("TREE", grid.best_params_, "CV", round(grid.best_score_, 4), "test", round(grid.score(X_test, y_test), 4))

clf = RandomizedSearchCV(
    LogisticRegression(solver="liblinear", max_iter=2000),
    LR_DIST, n_iter=N_ITER, random_state=RANDOM_STATE, cv=CV,
)
clf.fit(X_train, y_train)
print("LR  ", clf.best_params_, "CV", round(clf.best_score_, 4), "test", round(clf.score(X_test, y_test), 4))


## Simulation knobs

In [ ]:
N_ITER = 8
TRAIN_N = len(X_train)
FLIP_P = 0.0
print("edit the three names above and re-run the search cells")
print("current", N_ITER, TRAIN_N, FLIP_P)
